# Chapter 23: Nonlinear Least Squares

<a href="../lite/lab/index.html?path=ch23_nonlinear_least_squares.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def draw_cov_ellipse(ax, mean, cov, n_std=2, **kwargs):
    from matplotlib.patches import Ellipse
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1,1], vecs[0,1]))
    w, h = 2 * n_std * np.sqrt(np.maximum(vals, 0))
    ax.add_patch(Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs))

Nonlinear least squares is the workhorse of modern SLAM. Every time your phone
builds an AR scene or a self driving car refines its map, a Gauss-Newton solver
is running under the hood, iteratively tweaking thousands of variables until all
the measurements agree.

This chapter builds up the machinery: **residuals**, **Jacobians**, the
**Gauss-Newton** algorithm, **Levenberg-Marquardt** damping, and **robust cost
functions** for handling outliers.

## 23.1 Residual Minimization: The Cost Function

Given measurements $\mathbf{z}_i$ and a model $h_i(\mathbf{x})$, the
**residual** for measurement $i$ is:

$$r_i(\mathbf{x}) = \mathbf{z}_i - h_i(\mathbf{x})$$

We want to find $\mathbf{x}^*$ that minimizes the sum of squared residuals:

$$\mathbf{x}^* = \arg\min_{\mathbf{x}} \sum_{i} \| r_i(\mathbf{x}) \|^2
= \arg\min_{\mathbf{x}} \sum_{i} r_i(\mathbf{x})^T r_i(\mathbf{x})$$

If measurements have different uncertainties, we weight by the inverse covariance:

$$\mathbf{x}^* = \arg\min_{\mathbf{x}} \sum_{i} r_i^T \Omega_i \, r_i$$

where $\Omega_i = \Sigma_i^{-1}$ is the **information matrix** of measurement $i$.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
true_params = np.array([3.0, 4.0, 2.5])  # circle: cx, cy, radius
n_points = 30
sigma_noise = 0.3
# ──────────────────────────────────────────────────────────────────────────────

# Generate noisy points on a circle
theta_pts = np.linspace(0, 2*np.pi, n_points, endpoint=False)
cx, cy, R_true = true_params
pts_x = cx + R_true * np.cos(theta_pts) + np.random.randn(n_points) * sigma_noise
pts_y = cy + R_true * np.sin(theta_pts) + np.random.randn(n_points) * sigma_noise

# Residual: distance from point to circle surface
def circle_residuals(params, px, py):
    cx, cy, r = params
    dists = np.sqrt((px - cx)**2 + (py - cy)**2)
    return dists - r

# Evaluate cost at many points to visualize the cost surface
cx_grid = np.linspace(1, 5, 50)
cy_grid = np.linspace(2, 6, 50)
CX, CY = np.meshgrid(cx_grid, cy_grid)
cost_grid = np.zeros_like(CX)
for i in range(CX.shape[0]):
    for j in range(CX.shape[1]):
        res = circle_residuals([CX[i,j], CY[i,j], R_true], pts_x, pts_y)
        cost_grid[i,j] = np.sum(res**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(pts_x, pts_y, c='steelblue', s=30, zorder=5, label='Noisy data')
circle = plt.Circle((cx, cy), R_true, fill=False, color='forestgreen', lw=2, ls='--', label='True circle')
ax.add_patch(circle)
ax.plot(cx, cy, '+', color='forestgreen', ms=15, mew=2)
ax.set_aspect('equal'); ax.set_xlim(-1, 8); ax.set_ylim(-1, 8)
ax.set_title('Circle fitting: noisy data', fontsize=13)
ax.legend()

ax = axes[1]
cs = ax.contourf(CX, CY, np.log10(cost_grid), levels=20, cmap='Blues')
ax.plot(cx, cy, 'r*', ms=15, zorder=5, label='True center')
ax.set_xlabel('$c_x$'); ax.set_ylabel('$c_y$')
ax.set_title('Cost surface (log scale, radius fixed)', fontsize=13)
plt.colorbar(cs, ax=ax, label='$\\log_{10}$ cost')
ax.legend()

plt.tight_layout()
plt.show()

**Observation:** The cost surface has a clear minimum at the true circle center.
For a linear problem, we could solve for the minimum directly. For nonlinear
problems like this one, we need iterative methods.

## 23.2 Gauss-Newton: Linearize, Solve, Repeat

**Gauss-Newton** linearizes each residual around the current estimate:

$$r_i(\mathbf{x} + \Delta\mathbf{x}) \approx r_i(\mathbf{x}) + J_i \,\Delta\mathbf{x}$$

Substituting into the cost and setting the derivative to zero gives the
**normal equations**:

$$(J^T J) \, \Delta\mathbf{x} = -J^T \mathbf{r}$$

The algorithm:
1. Compute all residuals $r_i$ and the Jacobian $J$
2. Solve the normal equations for $\Delta\mathbf{x}$
3. Update $\mathbf{x} \leftarrow \mathbf{x} + \Delta\mathbf{x}$
4. Repeat until $\|\Delta\mathbf{x}\|$ is small

In [ ]:
def circle_jacobian(params, px, py):
    """Jacobian of circle residuals w.r.t. [cx, cy, r]."""
    cx, cy, r = params
    dx = px - cx
    dy = py - cy
    dists = np.sqrt(dx**2 + dy**2)
    dists = np.maximum(dists, 1e-10)  # avoid division by zero
    n = len(px)
    J = np.zeros((n, 3))
    J[:, 0] = -dx / dists   # d(residual)/d(cx)
    J[:, 1] = -dy / dists   # d(residual)/d(cy)
    J[:, 2] = -np.ones(n)   # d(residual)/d(r)
    return J

def gauss_newton_circle(x0, px, py, max_iter=20, tol=1e-8):
    """Gauss-Newton for circle fitting."""
    x = x0.copy()
    history = [x.copy()]
    costs = []
    
    for iteration in range(max_iter):
        r = circle_residuals(x, px, py)
        cost = np.sum(r**2)
        costs.append(cost)
        
        J = circle_jacobian(x, px, py)
        # Normal equations: (J^T J) dx = -J^T r
        JtJ = J.T @ J
        Jtr = J.T @ r
        dx = np.linalg.solve(JtJ, -Jtr)
        
        x = x + dx
        history.append(x.copy())
        
        if np.linalg.norm(dx) < tol:
            costs.append(np.sum(circle_residuals(x, px, py)**2))
            break
    
    return x, history, costs

print('Gauss-Newton circle fitter defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
x0_gn = np.array([1.0, 2.0, 1.0])  # initial guess (deliberately wrong)
max_iterations = 15
# ──────────────────────────────────────────────────────────────────────────────

x_gn, hist_gn, costs_gn = gauss_newton_circle(x0_gn, pts_x, pts_y, max_iterations)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(pts_x, pts_y, c='steelblue', s=30, zorder=3)
# Show iterations
colors_iter = plt.cm.Oranges(np.linspace(0.3, 1.0, len(hist_gn)))
for k, (h, c) in enumerate(zip(hist_gn, colors_iter)):
    circle_k = plt.Circle((h[0], h[1]), max(h[2], 0.1), fill=False,
                           color=c, lw=1.5, ls='--', alpha=0.7)
    ax.add_patch(circle_k)
    ax.plot(h[0], h[1], '+', color=c, ms=10, mew=2)

# Final
circle_final = plt.Circle((x_gn[0], x_gn[1]), x_gn[2], fill=False,
                           color='tomato', lw=3)
ax.add_patch(circle_final)
circle_true = plt.Circle((cx, cy), R_true, fill=False,
                          color='forestgreen', lw=2, ls=':')
ax.add_patch(circle_true)
ax.set_xlim(-2, 9); ax.set_ylim(-2, 9); ax.set_aspect('equal')
ax.set_title('Gauss-Newton iterations (orange to red)', fontsize=13)

ax = axes[1]
ax.semilogy(costs_gn, 'tomato', lw=2, marker='o', ms=6)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost $\\sum r_i^2$', fontsize=12)
ax.set_title('Convergence: cost drops rapidly', fontsize=13)

plt.tight_layout()
plt.show()

print(f'True:      cx={cx:.2f}, cy={cy:.2f}, R={R_true:.2f}')
print(f'Estimated: cx={x_gn[0]:.2f}, cy={x_gn[1]:.2f}, R={x_gn[2]:.2f}')
print(f'Converged in {len(costs_gn)} iterations')

**Observation:** Gauss-Newton converges quadratically near the solution. The
circle snaps from the initial guess to the correct fit in just a few
iterations. This fast convergence is why Gauss-Newton (and its variants)
dominate in SLAM backends.

## 23.3 Levenberg-Marquardt: Adaptive Damping

Gauss-Newton can fail when the linearization is poor (far from the solution).
**Levenberg-Marquardt (LM)** adds a damping term:

$$(J^T J + \lambda I) \, \Delta\mathbf{x} = -J^T \mathbf{r}$$

When $\lambda$ is large, the step is small and points toward the gradient
descent direction (safe but slow). When $\lambda$ is small, it behaves like
Gauss-Newton (fast but risky). The algorithm adapts $\lambda$:
- If the step reduces the cost, **decrease** $\lambda$ (trust Gauss-Newton more)
- If the step increases the cost, **increase** $\lambda$ (be more cautious)

In [ ]:
def levenberg_marquardt_circle(x0, px, py, max_iter=50, tol=1e-8):
    """Levenberg-Marquardt for circle fitting."""
    x = x0.copy()
    history = [x.copy()]
    costs = []
    lambdas = []
    
    r = circle_residuals(x, px, py)
    cost = np.sum(r**2)
    lam = 1.0  # initial damping
    
    for iteration in range(max_iter):
        costs.append(cost)
        lambdas.append(lam)
        
        J = circle_jacobian(x, px, py)
        JtJ = J.T @ J
        Jtr = J.T @ r
        
        # Damped normal equations
        dx = np.linalg.solve(JtJ + lam * np.eye(3), -Jtr)
        
        x_new = x + dx
        r_new = circle_residuals(x_new, px, py)
        cost_new = np.sum(r_new**2)
        
        if cost_new < cost:
            x = x_new
            r = r_new
            cost = cost_new
            lam *= 0.5  # trust GN more
            history.append(x.copy())
        else:
            lam *= 2.0  # be more cautious
        
        if np.linalg.norm(dx) < tol:
            break
    
    costs.append(cost)
    return x, history, costs, lambdas

print('Levenberg-Marquardt defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
x0_far = np.array([8.0, 8.0, 5.0])   # far initial guess (GN might struggle)
# ──────────────────────────────────────────────────────────────────────────────

# Compare GN and LM from a far initial guess
try:
    x_gn2, hist_gn2, costs_gn2 = gauss_newton_circle(x0_far, pts_x, pts_y, 30)
    gn_converged = True
except np.linalg.LinAlgError:
    gn_converged = False
    costs_gn2 = []

x_lm, hist_lm, costs_lm, lams = levenberg_marquardt_circle(x0_far, pts_x, pts_y, 50)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ax = axes[0]
ax.scatter(pts_x, pts_y, c='steelblue', s=30, zorder=3)
if gn_converged and len(hist_gn2) > 0:
    for h in hist_gn2[::2]:
        if h[2] > 0:
            c_patch = plt.Circle((h[0], h[1]), h[2], fill=False, color='steelblue', lw=1, alpha=0.4)
            ax.add_patch(c_patch)
for h in hist_lm[::2]:
    if h[2] > 0:
        c_patch = plt.Circle((h[0], h[1]), h[2], fill=False, color='orange', lw=1, alpha=0.5)
        ax.add_patch(c_patch)
# Final LM result
c_final = plt.Circle((x_lm[0], x_lm[1]), max(x_lm[2], 0.1), fill=False, color='tomato', lw=3)
ax.add_patch(c_final)
ax.set_xlim(-3, 12); ax.set_ylim(-3, 12); ax.set_aspect('equal')
ax.set_title('LM iterations (orange)', fontsize=13)

ax = axes[1]
if gn_converged and len(costs_gn2) > 0:
    ax.semilogy(costs_gn2, 'steelblue', lw=2, marker='o', ms=5, label='Gauss-Newton')
ax.semilogy(costs_lm, 'orange', lw=2, marker='s', ms=5, label='Levenberg-Marquardt')
ax.set_xlabel('Iteration'); ax.set_ylabel('Cost')
ax.set_title('Convergence comparison', fontsize=13)
ax.legend()

ax = axes[2]
ax.semilogy(lams, 'tomato', lw=2, marker='d', ms=5)
ax.set_xlabel('Iteration'); ax.set_ylabel('$\\lambda$')
ax.set_title('LM damping parameter $\\lambda$', fontsize=13)

plt.tight_layout()
plt.show()

print(f'LM result: cx={x_lm[0]:.2f}, cy={x_lm[1]:.2f}, R={x_lm[2]:.2f}')
print(f'True:      cx={cx:.2f}, cy={cy:.2f}, R={R_true:.2f}')

**Observation:** Levenberg-Marquardt starts cautiously (large $\lambda$) and
gradually reduces damping as it approaches the solution. It converges reliably
even from poor initial guesses where pure Gauss-Newton might oscillate or
diverge.

## 23.4 Robust Cost Functions: Handling Outliers

Least squares is sensitive to outliers because the squared cost $r^2$ gives
large residuals enormous influence. **Robust cost functions** (also called
**M-estimators**) reduce this influence.

**Huber cost:**
$$\rho(r) = \begin{cases} \frac{1}{2} r^2 & \text{if } |r| \leq \delta \\
\delta(|r| - \frac{1}{2}\delta) & \text{if } |r| > \delta \end{cases}$$

**Cauchy cost:**
$$\rho(r) = \frac{\delta^2}{2} \log\left(1 + \frac{r^2}{\delta^2}\right)$$

Both transition from quadratic (for small residuals) to linear or sublinear
(for large residuals), limiting outlier influence.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
delta_huber = 1.0
delta_cauchy = 1.0
# ──────────────────────────────────────────────────────────────────────────────

r_range = np.linspace(-5, 5, 200)

# Cost functions
squared = 0.5 * r_range**2
huber = np.where(np.abs(r_range) <= delta_huber,
                 0.5 * r_range**2,
                 delta_huber * (np.abs(r_range) - 0.5 * delta_huber))
cauchy = 0.5 * delta_cauchy**2 * np.log(1 + (r_range / delta_cauchy)**2)

# Weight functions (derivative of cost / r)
w_squared = np.ones_like(r_range)
w_huber = np.where(np.abs(r_range) <= delta_huber, 1.0,
                    delta_huber / np.maximum(np.abs(r_range), 1e-10))
w_cauchy = 1.0 / (1 + (r_range / delta_cauchy)**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(r_range, squared, 'steelblue', lw=2, label='Squared $\\frac{1}{2}r^2$')
ax.plot(r_range, huber, 'tomato', lw=2, label=f'Huber ($\\delta$={delta_huber})')
ax.plot(r_range, cauchy, 'orange', lw=2, label=f'Cauchy ($\\delta$={delta_cauchy})')
ax.set_xlabel('Residual $r$', fontsize=12)
ax.set_ylabel('Cost $\\rho(r)$', fontsize=12)
ax.set_title('Cost functions', fontsize=13)
ax.legend()
ax.set_ylim(0, 8)

ax = axes[1]
ax.plot(r_range, w_squared, 'steelblue', lw=2, label='Squared')
ax.plot(r_range, w_huber, 'tomato', lw=2, label='Huber')
ax.plot(r_range, w_cauchy, 'orange', lw=2, label='Cauchy')
ax.set_xlabel('Residual $r$', fontsize=12)
ax.set_ylabel('Weight $w(r)$', fontsize=12)
ax.set_title('Weight functions (outlier suppression)', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

**Key insight:** Huber transitions from quadratic to linear at $|r| = \delta$.
Cauchy is even gentler, approaching logarithmic growth. Both ensure that
outliers with large residuals contribute bounded influence to the optimization.

In [ ]:
def irls_circle(x0, px, py, cost_fn='squared', delta=1.0, max_iter=50, tol=1e-8):
    """Iteratively Reweighted Least Squares for circle fitting."""
    x = x0.copy()
    costs = []
    
    for iteration in range(max_iter):
        r = circle_residuals(x, px, py)
        
        # Compute weights based on cost function
        if cost_fn == 'squared':
            w = np.ones_like(r)
        elif cost_fn == 'huber':
            w = np.where(np.abs(r) <= delta, 1.0,
                          delta / np.maximum(np.abs(r), 1e-10))
        elif cost_fn == 'cauchy':
            w = 1.0 / (1 + (r / delta)**2)
        
        # Weighted cost
        costs.append(np.sum(w * r**2))
        
        J = circle_jacobian(x, px, py)
        W = np.diag(w)
        JtWJ = J.T @ W @ J
        JtWr = J.T @ W @ r
        
        dx = np.linalg.solve(JtWJ + 1e-6 * np.eye(3), -JtWr)
        x = x + dx
        
        if np.linalg.norm(dx) < tol:
            break
    
    return x, costs

print('IRLS circle fitter defined.')

---

## Capstone: Circle Fitting with Outliers

We add gross outliers to the circle data. Standard least squares fits a
wrong circle. Huber cost recovers the correct circle by downweighting
the outliers.

In [ ]:
# ── PARAMETERS ── change these and re-run ─────────────────────────────────────
np.random.seed(42)
n_inliers = 30
n_outliers = 8
sigma_inlier = 0.2
outlier_spread = 4.0
delta_robust = 0.5
# ──────────────────────────────────────────────────────────────────────────────

# Generate inlier points
theta_in = np.linspace(0, 2*np.pi, n_inliers, endpoint=False)
in_x = cx + R_true * np.cos(theta_in) + np.random.randn(n_inliers) * sigma_inlier
in_y = cy + R_true * np.sin(theta_in) + np.random.randn(n_inliers) * sigma_inlier

# Generate outliers (random positions)
out_x = cx + np.random.uniform(-outlier_spread, outlier_spread, n_outliers)
out_y = cy + np.random.uniform(-outlier_spread, outlier_spread, n_outliers)

all_x = np.concatenate([in_x, out_x])
all_y = np.concatenate([in_y, out_y])

x0_cap = np.array([2.0, 3.0, 2.0])  # initial guess

# Fit with squared cost
x_sq, costs_sq = irls_circle(x0_cap, all_x, all_y, 'squared')

# Fit with Huber cost
x_hub, costs_hub = irls_circle(x0_cap, all_x, all_y, 'huber', delta_robust)

# Fit with Cauchy cost
x_cau, costs_cau = irls_circle(x0_cap, all_x, all_y, 'cauchy', delta_robust)

fig, axes = plt.subplots(1, 3, figsize=(17, 6))

for ax, x_fit, label, color in [
    (axes[0], x_sq, 'Squared (standard)', 'steelblue'),
    (axes[1], x_hub, f'Huber ($\\delta$={delta_robust})', 'tomato'),
    (axes[2], x_cau, f'Cauchy ($\\delta$={delta_robust})', 'orange')]:
    
    ax.scatter(in_x, in_y, c='steelblue', s=30, zorder=3, label='Inliers')
    ax.scatter(out_x, out_y, c='tomato', s=50, marker='x', zorder=3, label='Outliers')
    c_true = plt.Circle((cx, cy), R_true, fill=False, color='forestgreen', lw=2, ls='--', label='True')
    ax.add_patch(c_true)
    c_fit = plt.Circle((x_fit[0], x_fit[1]), max(x_fit[2], 0.1), fill=False,
                        color=color, lw=3, label='Fitted')
    ax.add_patch(c_fit)
    err = np.linalg.norm(x_fit[:2] - [cx, cy])
    ax.set_xlim(-2, 9); ax.set_ylim(-2, 9); ax.set_aspect('equal')
    ax.set_title(f'{label}\ncenter error: {err:.3f} m', fontsize=12)
    ax.legend(fontsize=8)

plt.suptitle('Circle fitting: squared vs robust costs with outliers', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Show convergence
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(costs_sq, 'steelblue', lw=2, marker='o', ms=5, label='Squared')
ax.semilogy(costs_hub, 'tomato', lw=2, marker='s', ms=5, label='Huber')
ax.semilogy(costs_cau, 'orange', lw=2, marker='d', ms=5, label='Cauchy')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Weighted cost', fontsize=12)
ax.set_title('Convergence with outliers', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

print('Summary:')
print(f'  True center: ({cx:.2f}, {cy:.2f}), R={R_true:.2f}')
print(f'  Squared:     ({x_sq[0]:.2f}, {x_sq[1]:.2f}), R={x_sq[2]:.2f}  '
      f'center error={np.linalg.norm(x_sq[:2]-[cx,cy]):.3f}')
print(f'  Huber:       ({x_hub[0]:.2f}, {x_hub[1]:.2f}), R={x_hub[2]:.2f}  '
      f'center error={np.linalg.norm(x_hub[:2]-[cx,cy]):.3f}')
print(f'  Cauchy:      ({x_cau[0]:.2f}, {x_cau[1]:.2f}), R={x_cau[2]:.2f}  '
      f'center error={np.linalg.norm(x_cau[:2]-[cx,cy]):.3f}')

**Capstone observations:**
- Standard least squares is pulled toward the outliers, producing a biased fit.
- **Huber** cost downweights large residuals, recovering a fit close to the true circle.
- **Cauchy** cost is even more aggressive at suppressing outliers.
- In SLAM, outliers arise from incorrect data associations (wrong loop closures, mismatched features). Robust costs prevent these from corrupting the entire map.

---

## Exercises

### Exercise 23.1: Line fitting with Gauss-Newton

Implement Gauss-Newton for fitting a line $y = ax + b$ to noisy data.
Generate 20 points with noise. Show convergence from a wrong initial guess.

In [ ]:
# Your code here
# Residual: r_i = y_i - (a * x_i + b)
# Jacobian: J[i, 0] = -x_i, J[i, 1] = -1

### Exercise 23.2: Compare GN and LM convergence basins

For the circle fitting problem, try 20 random initial guesses spread over
a wide range. Count how many converge to the correct solution for GN vs LM.
Which method has a larger convergence basin?

In [ ]:
# Your code here

### Exercise 23.3: Tuning the Huber threshold (challenge)

With 30 inliers and 10 outliers, sweep the Huber threshold $\delta$ from
0.1 to 5.0. Plot the center estimation error vs $\delta$. What is the
optimal $\delta$? What happens when $\delta$ is too small or too large?

In [ ]:
# Your code here